In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# ציון אפשרויות Sampler

<Accordion>
<AccordionItem title="גרסאות חבילות">

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלה או בגרסאות חדשות יותר.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

אפשר להשתמש באפשרויות כדי להתאים אישית את הפרימיטיב Sampler. החלק הזה מתמקד בדרך לציין אפשרויות פרימיטיב של Qiskit Runtime. בעוד שהממשק של שיטת `run()` של הפרימיטיבים משותף לכל המימושים, האפשרויות שלהם אינן כך. עיין במקורות ה-API המתאימים לקבלת מידע על אפשרויות [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) ו-[`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## הגדרת אפשרויות Sampler
אפשר להגדיר אפשרויות בעת אתחול Sampler, אחרי אתחולו, או לעדכן את האפשרויות לאחר שה-Sampler אותחל. להוראות שימוש בטכניקות אלה, ראה את הנושא [מבוא לאפשרויות](/guides/runtime-options-overview#options-precedence).

בנוסף, אפשר להגדיר את ערך `shots` בשיטת `run()`, כפי שמתואר בסעיף הבא.

<span id="run-method"></span>
### שיטת Run()
הערכים היחידים שאפשר להעביר ל-`run()` הם אלה שמוגדרים בממשק. כלומר, `shots`. זה מחליף כל ערך שנקבע עבור `default_shots` בריצה הנוכחית.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### מקרים מיוחדים
<span id="shots"></span>
#### Shots
שיטת `SamplerV2.run` מקבלת שתי ארגומנטים: רשימה של PUBs, שכל אחד מהם יכול לציין ערך shots ספציפי ל-PUB, וארגומנט keyword של shots. ערכי ה-shots האלה הם חלק מממשק הביצוע של Sampler, ועצמאיים מאפשרויות ה-Runtime Sampler. הם גוברים על כל ערך שצוין כאפשרויות כדי לציית לאבסטרקציה של Sampler.

אם `shots` לא צוין על ידי אף PUB ולא בארגומנט keyword של הריצה (או שכולם `None`), אזי ישמש ערך ה-shots מהאפשרויות, בעיקר `default_shots`.

לסיכום, זה סדר העדיפות לציון shots ב-Sampler, עבור כל PUB מסוים:

1. אם ה-PUB מציין shots, השתמש בערך הזה.
2. אם ארגומנט keyword `shots` צוין ב-`run`, השתמש בערך הזה.
4. אם `twirling` מופעל (True כברירת מחדל), אזי משמש מכפלת `num_randomizations` ו-`shots_per_randomization`, כפי שצוינו כ-[אפשרויות `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options).
5. אם `sampler.options.default_shots` צוין, השתמש בערך הזה.

לכן, אם shots צוינו בכל המקומות האפשריים, ישמש זה בעל העדיפות הגבוהה ביותר (shots שצוינו ב-PUB).

דוגמה: